<a href="https://colab.research.google.com/github/umer-ateeq/GPT-Pretraining/blob/main/evaluate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluate ZeroToGPT 134M

Reproduce every perplexity in the README.

**Set the runtime to GPU first:** Runtime > Change runtime type > T4 GPU.
Then Runtime > Run all. Takes about 10 minutes.


In [ ]:
import torch
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print('GPU:', name if name else 'NONE. Runtime > Change runtime type > T4 GPU, then Run all again.')


## 1. Get the code


In [ ]:
import os
if not os.path.exists('/content/GPT-Pretraining'):
    !git clone -q https://github.com/umer-ateeq/GPT-Pretraining.git /content/GPT-Pretraining
%cd /content/GPT-Pretraining
!pip install -q tiktoken datasets transformers gdown huggingface_hub


## 2. Get the checkpoint and the held-out split

The weights are 538 MB, from [umerateeq/zerotogpt-134m](https://huggingface.co/umerateeq/zerotogpt-134m).


In [ ]:
import os
if not os.path.exists('weights8b_300epoch.pth'):
    from huggingface_hub import hf_hub_download
    hf_hub_download('umerateeq/zerotogpt-134m', 'weights8b_300epoch.pth', local_dir='.')
if not os.path.exists('validation.bin'):
    !gdown -q '1kowxuffn3VRKGnERvSBWTTxaGULGqu9q'
print(sorted(f for f in os.listdir('.') if f.endswith(('.pth', '.bin'))))


## 3. Score every dataset

Held-out FineWeb-Edu, TinyStories and WikiText-2, at a 128-token window.
A markdown table is printed at the end, ready to paste into the README.


In [ ]:
!python evaluate.py --ckpt weights8b_300epoch.pth --all --data-bin validation.bin


## 4. Harness calibration

GPT-2-small through the identical scoring function. Its published WikiText-2 perplexity
is about 29.4 at its full context, so a sane figure here means the harness is sound.


In [ ]:
!python evaluate.py --model gpt2 --mode wikitext
